# Week 8: Vector Workflows in Python

This notebook replicates QGIS vector operations using GeoPandas:
- Load and clean spatial data
- Perform spatial joins
- Calculate density metrics
- Create choropleth maps

---

## Before you start: Where is this notebook running?

This notebook can run in two places:

| Environment | What it means | Your data files are... |
|-------------|---------------|------------------------|
| **Google Colab** | Runs on Google's servers in your browser | On your Google Drive |
| **Jupyter (local)** | Runs on your own computer | On your computer's hard drive |

**The key insight:** When you open this notebook in Colab, the code runs on a Google computer in a data center somewhere—not on your laptop. That Google computer can't see your laptop's files. So we need to connect it to your Google Drive, where you'll store your data files.

---

## Step 0: Set up your environment

Run this cell first. It:
1. Detects whether you're in Colab or local Jupyter
2. Installs required packages (Colab only)
3. Sets up the correct path to your data

In [ ]:
# Detect environment and install packages
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing GIS packages (takes ~1 minute)...")
    !pip install geopandas contextily mapclassify -q
    print("Done!")
else:
    print("Running in local Jupyter")
    print("Make sure you activated your conda environment: conda activate intro-gis")

---

## Step 1: Connect to your data

### If you're using Google Colab: Mount your Google Drive

**What does "mounting" mean?**

Think of Google Drive as a USB drive. When you plug a USB into your computer, it "mounts"—your computer can suddenly see and access the files on it. 

"Mounting" Google Drive in Colab does the same thing: it connects your Drive to the Colab computer so your code can read files from Drive.

**Before running the cell below:**

1. Open [Google Drive](https://drive.google.com) in another tab
2. Create a folder called `intro-gis` (if you haven't already)
3. Inside `intro-gis`, create `data/raw/` and `data/processed/`
4. Upload your data files to `data/raw/`:
   - `neighbourhoods.geojson`
   - `incidents.geojson`

Your Drive should look like:
```
My Drive/
└── intro-gis/
    └── data/
        ├── raw/              ← Upload files here
        │   ├── neighbourhoods.geojson
        │   └── incidents.geojson
        └── processed/        ← Your outputs go here
```

### If you're using local Jupyter:

Make sure your files are in `intro-gis/data/raw/` on your computer, and you launched Jupyter from the `intro-gis` folder.

In [ ]:
from pathlib import Path

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Set paths to your data folders in Drive
    RAW = Path("/content/drive/MyDrive/intro-gis/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/data/processed")
    print(f"Raw data folder: {RAW}")
    print(f"Processed folder: {PROCESSED}")
else:
    # Local paths (relative to notebook location)
    RAW = Path("../data/raw")
    PROCESSED = Path("../data/processed")
    print(f"Raw data folder: {RAW.resolve()}")
    print(f"Processed folder: {PROCESSED.resolve()}")

# Check if the folders exist
if RAW.exists():
    print("\nRaw folder found! Files:", list(RAW.glob("*")))
else:
    print("\nRaw folder NOT found. Check that you created data/raw/ and uploaded files.")

# Create processed folder if it doesn't exist
PROCESSED.mkdir(parents=True, exist_ok=True)

---

## Step 2: Import libraries

These are the Python tools we'll use.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

---

## Step 3: Load your data

This is equivalent to `Layer > Add Vector Layer` in QGIS.

In [ ]:
# Load the data files
neighbourhoods = gpd.read_file(RAW / "neighbourhoods.geojson")
incidents = gpd.read_file(RAW / "incidents.geojson")

print(f"Loaded {len(neighbourhoods)} neighbourhoods")
print(f"Loaded {len(incidents)} incidents")

# Preview the neighbourhoods (like opening the attribute table)
neighbourhoods.head()

---

## Step 4: Clean and prepare data

Standardize column names and calculate area in km².

In [ ]:
# Convert column names to lowercase (avoids case-sensitivity issues)
neighbourhoods = neighbourhoods.rename(columns=str.lower)
incidents = incidents.rename(columns=str.lower)

# Calculate area in km² (need to project to meters first)
neighbourhoods["area_km2"] = neighbourhoods.to_crs(3857).area / 1e6

print("Columns:", list(neighbourhoods.columns))
neighbourhoods[["sa2_name21", "area_km2"]].head()

---

## Step 5: Spatial join

Count how many incidents fall within each neighbourhood. This is equivalent to:
- QGIS: `Vector > Data Management > Join Attributes by Location (Summary)`

In [ ]:
# Spatial join: link each incident to its neighbourhood
joined = gpd.sjoin(incidents, neighbourhoods, predicate="within", how="left")

# Count incidents per neighbourhood
counts = joined.groupby("sa2_name21").size().rename("incident_count")

# Merge counts back to neighbourhoods
neighbourhoods = neighbourhoods.merge(counts, on="sa2_name21", how="left")
neighbourhoods["incident_count"] = neighbourhoods["incident_count"].fillna(0)

print("Top 5 neighbourhoods by incident count:")
neighbourhoods.nlargest(5, "incident_count")[["sa2_name21", "incident_count"]]

---

## Step 6: Calculate incident rate

Incidents per square kilometre (normalizes for area size).

In [ ]:
neighbourhoods["rate_per_km2"] = neighbourhoods["incident_count"] / neighbourhoods["area_km2"]

print("Top 5 neighbourhoods by incident RATE:")
neighbourhoods.nlargest(5, "rate_per_km2")[["sa2_name21", "incident_count", "area_km2", "rate_per_km2"]]

---

## Step 7: Create a choropleth map

Visualize the incident rate. This is equivalent to graduated symbology in QGIS.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

neighbourhoods.plot(
    column="rate_per_km2",
    scheme="quantiles",       # Classification method (like QGIS Mode)
    k=5,                       # Number of classes
    cmap="YlOrRd",            # Color ramp
    legend=True,
    legend_kwds={"title": "Rate per km²"},
    ax=ax
)

ax.set_title("Incident Rate by Neighbourhood", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

---

## Step 8: Export for QGIS

Save your results to `data/processed/` as a GeoPackage so you can open it in QGIS for final cartography.

In [ ]:
# Create output path
output_path = PROCESSED / "week08_neighbourhoods_summary.gpkg"

# Save to GeoPackage
neighbourhoods.to_file(output_path, driver="GPKG")

print(f"Saved to: {output_path}")
print("You can now open this file in QGIS!")

---

## Done!

You've completed a full vector analysis workflow in Python:

1. Connected to your data (Drive or local)
2. Loaded spatial data with GeoPandas
3. Cleaned and prepared attributes
4. Performed a spatial join
5. Calculated density metrics
6. Created a choropleth visualization
7. Exported results for QGIS

**Save your work:**
- **Notebook:** Save this notebook
  - Colab: `File > Save a copy in Drive`
  - Local: `Ctrl+S` or `Cmd+S`
- **Output file created:** `data/processed/week08_neighbourhoods_summary.gpkg`
  - This file contains your neighbourhood analysis with incident counts and rates
  - Open it in QGIS for final map production!